# 08 · Hiperparámetros con Optuna (etapa 7)

**Ficha:** `fichas/etapa_7_hiperparametros.md` (las seis decisiones van con la recomendación).

1. **Optuna** busca hiperparámetros de LightGBM con las variables de `fe02` (`B2+B3+B4+B5`, base V5, 507 variables), optimizando la ganancia en la meseta de la prueba **A** (entrena `202103`–`202104`, predice `202106` completo).
   - Para que cada prueba sea rápida, entrena con **todas las bajas y el 20 % de los `CONTINUA`** (las mismas filas en todas las pruebas).
   - La prueba 0 es `z470`, cargada a mano.
2. **Validación:** las 3 mejores, con **todos los datos** y `min_data_in_leaf` ÷ 0,2 (el truco de `z494`), 5 semillas, en A y C. Se comparan semilla por semilla contra `z470` (las curvas del `05`).
3. **Elección** (la decide el usuario): `ELEGIDA` = el número de prueba. Se escribe `hiperparametros.json` para el `03`.

`MODO_RAPIDO = True`: 3 pruebas en un estudio aparte (`..._rapido`, no se mezcla con el de verdad) y la validación de la mejor con 1 semilla, solo en A. Sirve para medir tiempos.

## 1. Parámetros

In [22]:
from pathlib import Path
import json
import time
import warnings

import duckdb
import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
from scipy.stats import ttest_rel, wilcoxon

warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')  # falso positivo pandas 2.2 + Py 3.14
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Parámetros -------------------------------------------------------------
MODO_RAPIDO = False              # True: 3 pruebas (estudio aparte) · False: N_PRUEBAS
N_PRUEBAS = 50                  # decisión 4 (si a la mitad todavía mejora, se suben: el estudio se retoma)
ELEGIDA = 44                  # <- después de la validación: el número de prueba elegida

CONFIG, BASE, BLOQUES = 'z470_bajas', 'V5', 'B2+B3+B4+B5'      # las variables de fe02
PRUEBA_OPTUNA = 'A'             # decisión 1
FRACCION = 0.2                  # decisión 2: fracción de CONTINUA en Optuna
TOP, SEMILLAS_VAL, PRUEBAS_VAL = 3, 5, ['A', 'C']                # decisión 5
SEMILLA_PRIMIGENIA = 979201
PRUEBA_C = {'train': [202103, 202104, 202105], 'test': 202106}

ESTUDIO = 'e7_fe02_A_us20' + ('_rapido' if MODO_RAPIDO else '')
EXP02, EXP05 = Path('../exp/02_linea_base'), Path('../exp/05_variables')
EXP = Path('../exp/08_optuna')
CURVAS = EXP / 'curvas'
STORAGE = f"sqlite:///{(EXP / ('optuna_e7' + ('_rapido' if MODO_RAPIDO else '') + '.db')).as_posix()}"
LARGO_CURVA = 25_000
N_ENVIOS = 12_000               # el N decidido: también se reporta la ganancia ahí
# -----------------------------------------------------------------------------

CURVAS.mkdir(parents=True, exist_ok=True)
config02 = json.loads((EXP02 / 'configuraciones.json').read_text())
PRUEBAS = {**config02['pruebas'], 'C': PRUEBA_C}
VENTANA_SUAVIZADO = config02['ventana_suavizado']
GANANCIA_ACIERTO, COSTO_ESTIMULO = config02['ganancia_acierto'], config02['costo_estimulo']
SEMILLAS = json.loads((EXP02 / 'semillas.json').read_text())['semillas']
HIPER, CLASE = CONFIG.split('_', 1)
POSITIVOS = config02['clases'][CLASE]
Z470 = dict(config02['hiperparametros'][HIPER])

bloques_05 = json.loads((EXP05 / 'bloques.json').read_text())
assert bloques_05['base'] == BASE and bloques_05['config'] == CONFIG and BLOQUES in bloques_05['variables']
VARIABLES = bloques_05['variables'][BLOQUES]
FUENTES, NA_EN_MES = bloques_05['fuentes'], bloques_05['na_en_mes']
FE_PARQUET = Path(bloques_05['fe_parquet'])
if MODO_RAPIDO:
    N_PRUEBAS, TOP, SEMILLAS_VAL, PRUEBAS_VAL = 3, 1, 1, ['A']
print(f'>>> estudio {ESTUDIO} · {N_PRUEBAS} pruebas · {len(VARIABLES)} variables · validación: top {TOP}, '
      f'{SEMILLAS_VAL} semillas, {PRUEBAS_VAL}')

>>> estudio e7_fe02_A_us20 · 50 pruebas · 507 variables · validación: top 3, 5 semillas, ['A', 'C']


## 2. Datos

Las mismas variables y la misma limpieza que el `05` (la lista exacta de `bloques.json`). Se cargan solo las columnas necesarias.

In [23]:
t0 = time.time()
columnas = ['numero_de_cliente', 'foto_mes', 'clase_ternaria'] + VARIABLES
lista = ', '.join(f'"{c}"' for c in columnas)
df = duckdb.sql(f"select {lista} from read_parquet('{FE_PARQUET.as_posix()}')").df()
print(f'{len(df):,} filas · {len(VARIABLES)} variables · {time.time() - t0:.0f} s')


def matriz(indices):
    X = df[VARIABLES].iloc[indices].copy()
    meses = df['foto_mes'].iloc[indices].to_numpy()
    for columna, meses_na in NA_EN_MES.items():
        afectadas = [v for v in VARIABLES if v == columna or columna in FUENTES.get(v, [])]
        X.loc[np.isin(meses, meses_na), afectadas] = np.nan
    return X


def particion(prueba):
    meses = PRUEBAS[prueba]
    es_train = df['foto_mes'].isin(meses['train']) & df['clase_ternaria'].notna()
    return np.flatnonzero(es_train), np.flatnonzero(df['foto_mes'] == meses['test'])


def curva_ganancia(prob, es_baja2, largo=LARGO_CURVA):
    orden = np.argsort(-prob, kind='stable')
    return np.cumsum(np.where(es_baja2[orden], GANANCIA_ACIERTO, -COSTO_ESTIMULO))[:largo].astype(np.float64)


def meseta(curva):
    s = pd.Series(curva).rolling(VENTANA_SUAVIZADO, center=True, min_periods=VENTANA_SUAVIZADO).mean().to_numpy()
    i = int(np.nanargmax(s))
    return i + 1, float(s[i])


# Submuestreo fijo para Optuna: todas las bajas y FRACCION de los CONTINUA (control 2)
i_train, i_test = particion(PRUEBA_OPTUNA)
clase_train = df['clase_ternaria'].iloc[i_train].to_numpy()
rng = np.random.default_rng(SEMILLA_PRIMIGENIA)
queda = (clase_train != 'CONTINUA') | (rng.random(len(i_train)) < FRACCION)
i_us = i_train[queda]
y_us = df['clase_ternaria'].iloc[i_us].isin(POSITIVOS).astype(int).to_numpy()
X_test = matriz(i_test)
es_baja2_test = (df['clase_ternaria'].iloc[i_test] == 'BAJA+2').to_numpy()
assert y_us.sum() == np.isin(clase_train, POSITIVOS).sum(), 'Tienen que quedar todas las bajas'
assert len(i_test) == 164_114, 'El mes de prueba va completo (control 3)'
dtrain_us = lgb.Dataset(matriz(i_us), label=y_us, params={'max_bin': 31, 'feature_pre_filter': False},
                        free_raw_data=False)
print(f'Optuna entrena con {len(i_us):,} filas ({y_us.sum():,} bajas + {(~np.isin(clase_train[queda], POSITIVOS)).sum():,} CONTINUA) '
      f'y evalúa en {len(i_test):,} filas de {PRUEBAS[PRUEBA_OPTUNA]["test"]}')

983,061 filas · 507 variables · 3 s
Optuna entrena con 68,229 filas (4,082 bajas + 64,147 CONTINUA) y evalúa en 164,114 filas de 202106


## 3. Optuna

- **Fijos:** `gbdt`, `max_bin` 31, `bagging_freq` 1, sin regularizaciones (Clase 08), `feature_pre_filter` en `False` (necesario para cambiar `min_data_in_leaf` sin rearmar el dataset).
- **Espacio** (decisión 3): `num_leaves` 8–512, `min_data_in_leaf` 10–2.000 (escala submuestreada), `learning_rate` 0,01–0,1 y `num_iterations` 200–1.500, todos logarítmicos; `feature_fraction` 0,2–1,0 y `bagging_fraction` 0,4–1,0.
  - El piso de `min_data_in_leaf` baja de 20 (ficha) a 10, para que entre `z470`: sus 76 a escala completa son ~15 en la escala submuestreada.
- **Prueba 0:** `z470` (8 hojas, `min_data_in_leaf` 15, 0,027, 1.000 árboles, 0,8, sin bagging).
- **Una semilla de LightGBM** en todas las pruebas, la primera de las de backtest. El sampler TPE usa la semilla primigenia. El estudio se retoma si se corta.

In [24]:
FIJOS = {'objective': 'binary', 'boosting': 'gbdt', 'max_bin': 31, 'bagging_freq': 1,
         'lambda_l1': 0.0, 'lambda_l2': 0.0, 'min_gain_to_split': 0.0,
         'feature_pre_filter': False, 'verbose': -1}
SEMILLA_MODELO = SEMILLAS[0]


def objetivo(trial):
    p = {**FIJOS,
         'num_leaves': trial.suggest_int('num_leaves', 8, 512, log=True),
         'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 2000, log=True),
         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
         'feature_fraction': trial.suggest_float('feature_fraction', 0.2, 1.0),
         'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
         'seed': SEMILLA_MODELO}
    rondas = trial.suggest_int('num_iterations', 200, 1500, log=True)
    t0 = time.time()
    modelo = lgb.train(p, dtrain_us, num_boost_round=rondas)
    n, g = meseta(curva_ganancia(modelo.predict(X_test), es_baja2_test))
    trial.set_user_attr('N_meseta', n)
    trial.set_user_attr('segundos', round(time.time() - t0))
    print(f'prueba {trial.number}: {g / 1e6:,.1f} M (N {n:,}) · hojas {p["num_leaves"]} · mdl {p["min_data_in_leaf"]} · '
          f'lr {p["learning_rate"]:.3f} · árboles {rondas} · ff {p["feature_fraction"]:.2f} · bf {p["bagging_fraction"]:.2f} · '
          f'{time.time() - t0:.0f} s')
    return g


estudio = optuna.create_study(study_name=ESTUDIO, storage=STORAGE, direction='maximize', load_if_exists=True,
                              sampler=optuna.samplers.TPESampler(seed=SEMILLA_PRIMIGENIA))
if len(estudio.trials) == 0:     # prueba 0: z470
    estudio.enqueue_trial({'num_leaves': 8, 'min_data_in_leaf': 15, 'learning_rate': 0.027, 'num_iterations': 1000,
                           'feature_fraction': 0.8, 'bagging_fraction': 1.0})
hechas = len([t for t in estudio.trials if t.state == optuna.trial.TrialState.COMPLETE])
t_total = time.time()
estudio.optimize(objetivo, n_trials=max(0, N_PRUEBAS - hechas))
print(f'{len(estudio.trials)} pruebas en el estudio · esta corrida: {(time.time() - t_total) / 60:.1f} min')

50 pruebas en el estudio · esta corrida: 0.0 min


## 4. Las mejores pruebas

Control 1: la prueba 0 (`z470` con submuestreo) tiene que dar del orden de la meseta de `B2+B3+B4+B5` en A del `05` (~368 M), sin ser igual.

In [25]:
pruebas = estudio.trials_dataframe(attrs=('number', 'value', 'params', 'user_attrs', 'state'))
pruebas = pruebas[pruebas['state'] == 'COMPLETE'].rename(columns=lambda c: c.replace('params_', '').replace('user_attrs_', ''))
pruebas['ganancia_M'] = pruebas['value'] / 1e6
pruebas = pruebas.sort_values('value', ascending=False)
pruebas.drop(columns=['value', 'state']).to_csv(EXP / ('pruebas_rapido.csv' if MODO_RAPIDO else 'pruebas.csv'), index=False)
print(f"Prueba 0 (z470 con submuestreo): {pruebas.set_index('number').loc[0, 'ganancia_M']:,.1f} M")
mejores = pruebas.head(TOP)['number'].tolist()
print('Las mejores:', mejores)
pruebas.head(10).drop(columns=['value', 'state']).round(3)

Prueba 0 (z470 con submuestreo): 365.6 M
Las mejores: [28, 42, 44]


,number,bagging_fraction,feature_fraction,learning_rate,min_data_in_leaf,num_iterations,num_leaves,N_meseta,segundos,ganancia_M
28,28,0.881,0.624,0.051,311,547,20,11504,6,414.374
42,42,0.929,0.403,0.042,371,637,36,10358,13,405.395
44,44,0.822,0.686,0.051,464,397,32,12020,6,403.842
34,34,0.894,0.710,0.026,441,562,35,13867,9,402.112
31,31,0.789,0.675,0.047,333,555,21,12313,6,399.694
45,45,0.829,0.827,0.051,394,379,30,12866,5,399.018
17,17,0.745,0.463,0.015,72,787,48,11704,18,398.954
24,24,0.916,0.369,0.036,115,731,57,14344,18,396.392
41,41,0.893,0.276,0.034,1058,761,51,12974,17,396.201
29,29,0.789,0.715,0.049,376,564,21,11378,6,396.036


## 5. Validación: con todos los datos, contra `z470`

Para cada una de las mejores: todos los datos de entrenamiento, `min_data_in_leaf` ÷ `FRACCION`, `SEMILLAS_VAL` semillas, en `PRUEBAS_VAL`. `z470` se lee del `05` (mismas semillas).

**Control 6:** `z470` entrenado acá con la primera semilla tiene que dar **exactamente** la curva del `05`. Si da distinto, algo cambió en los datos.

Con 5 semillas, el Wilcoxon no puede bajar de p = 0,0625. Por eso se informa también el **t pareado**, y la regla usa ese.

In [26]:
def params_completos(numero):
    p = estudio.trials[numero].params
    return {**FIJOS, 'num_leaves': p['num_leaves'], 'min_data_in_leaf': int(round(p['min_data_in_leaf'] / FRACCION)),
            'learning_rate': p['learning_rate'], 'feature_fraction': p['feature_fraction'],
            'bagging_fraction': p['bagging_fraction'], 'num_iterations': p['num_iterations']}


def entrenar_completo(params, prueba, semilla):
    i_tr, i_te = particion(prueba)
    y = df['clase_ternaria'].iloc[i_tr].isin(POSITIVOS).astype(int).to_numpy()
    p = dict(params)
    rondas = p.pop('num_iterations')
    modelo = lgb.train({**p, 'seed': semilla}, lgb.Dataset(matriz(i_tr), label=y), num_boost_round=rondas)
    return curva_ganancia(modelo.predict(matriz(i_te)), (df['clase_ternaria'].iloc[i_te] == 'BAJA+2').to_numpy())


def curva_z470(prueba, semilla):
    return np.load(EXP05 / 'curvas' / f'{CONFIG}_{BASE}_{BLOQUES}_{prueba}_s{semilla}.npz')['curva']


# Control 6
t0 = time.time()
dif = meseta(entrenar_completo(Z470, 'A', SEMILLAS[0]))[1] - meseta(curva_z470('A', SEMILLAS[0]))[1]
print(f'Control 6: z470 en A, semilla {SEMILLAS[0]}, contra el 05: diferencia {dif:,.0f} · {time.time() - t0:.0f} s')
assert abs(dif) < 1, 'z470 no reproduce la curva del 05'

curvas_val = {}
for numero in mejores:
    for prueba in PRUEBAS_VAL:
        for semilla in SEMILLAS[:SEMILLAS_VAL]:
            archivo = CURVAS / f'{ESTUDIO}_p{numero}_{prueba}_s{semilla}.npy'
            if not archivo.exists():
                t0 = time.time()
                np.save(archivo, entrenar_completo(params_completos(numero), prueba, semilla))
                print(f'prueba {numero} · {prueba} · semilla {semilla} · {time.time() - t0:.0f} s')
            curvas_val[(numero, prueba, semilla)] = np.load(archivo)

Control 6: z470 en A, semilla 294103, contra el 05: diferencia 0 · 20 s


## 6. Comparación pareada contra `z470`

Por prueba: la meseta de cada semilla, la diferencia contra `z470` (media, cuántas semillas mejoran, t pareado y Wilcoxon) y la ganancia en N = 12.000.

**Regla** (decisión 6): se adopta si **A + C**, sumando semilla por semilla, le gana a `z470`, y **en ninguna de las dos pierde** con p < 0,05 (t pareado).

In [27]:
filas = []
semillas_v = SEMILLAS[:SEMILLAS_VAL]
for numero in mejores:
    fila, difs = {'prueba': numero, **{k: v for k, v in params_completos(numero).items() if k not in FIJOS}}, {}
    for prueba in PRUEBAS_VAL:
        g_opt = np.array([meseta(curvas_val[(numero, prueba, s)])[1] for s in semillas_v])
        g_z = np.array([meseta(curva_z470(prueba, s))[1] for s in semillas_v])
        d = g_opt - g_z
        difs[prueba] = d
        fila[f'{prueba}_optuna'] = g_opt.mean() / 1e6
        fila[f'{prueba}_z470'] = g_z.mean() / 1e6
        fila[f'{prueba}_dif'] = d.mean() / 1e6
        fila[f'{prueba}_mejoran'] = f'{(d > 0).sum()}/{len(d)}'
        fila[f'{prueba}_p_t'] = ttest_rel(g_opt, g_z).pvalue if len(d) > 1 else np.nan
        fila[f'{prueba}_p_wilcoxon'] = wilcoxon(d).pvalue if len(d) >= 5 and np.any(d != 0) else np.nan
        fila[f'{prueba}_en_12000'] = np.mean([curvas_val[(numero, prueba, s)][N_ENVIOS - 1] for s in semillas_v]) / 1e6
        fila[f'{prueba}_z470_en_12000'] = np.mean([curva_z470(prueba, s)[N_ENVIOS - 1] for s in semillas_v]) / 1e6
    total = sum(difs.values())
    pierde = any(difs[p].mean() < 0 and fila[f'{p}_p_t'] < 0.05 for p in PRUEBAS_VAL)
    fila['adoptar'] = bool(total.mean() > 0 and not pierde and SEMILLAS_VAL > 1)
    filas.append(fila)

validacion = pd.DataFrame(filas)
validacion.to_csv(EXP / ('validacion_rapido.csv' if MODO_RAPIDO else 'validacion.csv'), index=False)
print('Pasan la regla:', validacion.query('adoptar')['prueba'].tolist() or 'ninguna', '(la decisión es del usuario)')
validacion.round(3).T

Pasan la regla: [28, 42, 44] (la decisión es del usuario)


,0,1,2
prueba,28,42,44
num_leaves,20,36,32
min_data_in_leaf,1555,1855,2320
learning_rate,0.051,0.042,0.051
feature_fraction,0.624,0.403,0.686
bagging_fraction,0.881,0.929,0.822
num_iterations,547,637,397
A_optuna,380.436,398.655,394.163
A_z470,370.811,370.811,370.811
A_dif,9.626,27.844,23.352


## 7. Elección y `hiperparametros.json`

Con `ELEGIDA` = el número de prueba, se escribe `../exp/08_optuna/hiperparametros_p<ELEGIDA>.json` (uno por prueba, para que elegir otra no pise la anterior) con los hiperparámetros **para todos los datos** (`min_data_in_leaf` ya corregido), que lee el `03`. Con `None`, no se escribe nada.

In [28]:
if ELEGIDA is None:
    print('ELEGIDA = None: todavía no se eligió (mirar la validación).')
else:
    assert not MODO_RAPIDO, 'La elección se hace con la corrida completa'
    assert ELEGIDA in mejores, f'ELEGIDA tiene que ser una de las validadas: {mejores}'
    salida = {'estudio': ESTUDIO, 'prueba': int(ELEGIDA), 'config': CONFIG, 'base': BASE, 'bloques': BLOQUES,
              'fraccion_optuna': FRACCION, 'params': params_completos(ELEGIDA),
              'validacion': validacion.set_index('prueba').loc[ELEGIDA].to_dict()}
    archivo = EXP / f'hiperparametros_p{ELEGIDA}.json'          # uno por prueba: no se pisan
    archivo.write_text(json.dumps(salida, indent=2, default=str))
    print(f'Escrito {archivo}:', salida['params'])

Escrito ../exp/08_optuna/hiperparametros_p44.json: {'objective': 'binary', 'boosting': 'gbdt', 'max_bin': 31, 'bagging_freq': 1, 'lambda_l1': 0.0, 'lambda_l2': 0.0, 'min_gain_to_split': 0.0, 'feature_pre_filter': False, 'verbose': -1, 'num_leaves': 32, 'min_data_in_leaf': 2320, 'learning_rate': 0.0508586909246285, 'feature_fraction': 0.6859223244067804, 'bagging_fraction': 0.8218454720600575, 'num_iterations': 397}
